This notebook is to explore the results of the model derived INT

In [1]:
# Imports
import h5py
import numpy as np

In [2]:
# Load the data

data_files = {
    'Dopamine': '/home/frank/HBNM/experiments/INTs/INT_Results/INT_Dopamine_AVG_heterogeneous_iter28_20250816_2006.hdf5',
    'GABAa': '/home/frank/HBNM/experiments/INTs/INT_Results/INT_GABAa_AVG_heterogeneous_iter33_20250816_2037.hdf5',
    'Heterogeneous': '/home/frank/HBNM/experiments/INTs/INT_Results/INT_heterogeneous_iter35_20250816_1839.hdf5',
    'Homogeneous': '/home/frank/HBNM/experiments/INTs/INT_Results/INT_homogeneous_iter20_20250816_1953.hdf5',
    'NMDA': '/home/frank/HBNM/experiments/INTs/INT_Results/INT_NMDA_AVG_heterogeneous_iter41_20250816_2053.hdf5'
}

In [3]:
# Initialize dictionary to store all timescales
all_timescales = {}

# Loop through each data file
for model_name, file_path in data_files.items():
    region_timescales = []
    
    # Open the HDF5 file
    with h5py.File(file_path, 'r') as f:
        # Loop through all 180 regions
        for region_num in range(180):
            # Format region number with leading zeros
            region_key = f'fitting_results/regions/region_{region_num:03d}'
            g = f[region_key]
            # Get and store the timescale
            region_timescales.append(g.attrs['timescale'])
    
    # Store timescales for this model as numpy array
    all_timescales[model_name] = np.array(region_timescales)

# Print results for verification
for model_name, timescales in all_timescales.items():
    print(f"{model_name} timescales (first 5): {timescales[:5]}")


Dopamine timescales (first 5): [0.16489228 0.57103136 0.408393   0.05       0.2242044 ]
GABAa timescales (first 5): [0.05248998 1.52964094 0.3969359  0.18347995 0.27749339]
Heterogeneous timescales (first 5): [3.24828307 0.3784638  0.28215417 0.16355365 0.22519829]
Homogeneous timescales (first 5): [0.30098311 0.67646855 0.41263901 0.05       0.35079177]
NMDA timescales (first 5): [9.69225591e-02 2.91468464e+00 1.64442416e+01 1.52871404e-01
 1.81475195e+02]


In [4]:
all_timescales.keys()

dict_keys(['Dopamine', 'GABAa', 'Heterogeneous', 'Homogeneous', 'NMDA'])

### Mapping

Link: https://github.com/ColeLab/ColeAnticevicNetPartition

In [5]:
# Load the network label file into a dictionary
labelfile = '/home/frank/HBNM/data/network_labelfile.txt'

network_mapping = {}

with open(labelfile, 'r') as f:
    lines = [line.strip() for line in f.readlines() if line.strip()]

# Process pairs of lines (network name, then ID + RGBA values)
for i in range(0, len(lines), 2):
    if i + 1 < len(lines):
        network_name = lines[i]
        values = lines[i + 1].split()
        
        if len(values) >= 5:  # ID + 4 RGBA values
            network_id = int(values[0])
            rgba = [int(values[j]) for j in range(1, 5)]  # R, G, B, A
            
            network_mapping[network_name] = {
                'id': network_id,
                'rgba': rgba,
                'rgb': rgba[:3]  # Just RGB without alpha
            }

# Display the result
for name, info in network_mapping.items():
    print(f"{name}: ID={info['id']}, RGB={info['rgb']}, RGBA={info['rgba']}")

Visual1: ID=1, RGB=[0, 0, 255], RGBA=[0, 0, 255, 255]
Visual2: ID=2, RGB=[100, 0, 255], RGBA=[100, 0, 255, 255]
Somatomotor: ID=3, RGB=[0, 255, 255], RGBA=[0, 255, 255, 255]
Cingulo-Opercular: ID=4, RGB=[153, 0, 153], RGBA=[153, 0, 153, 255]
Language: ID=6, RGB=[0, 154, 154], RGBA=[0, 154, 154, 255]
Default: ID=9, RGB=[255, 0, 0], RGBA=[255, 0, 0, 255]
Frontoparietal: ID=7, RGB=[255, 255, 0], RGBA=[255, 255, 0, 255]
Auditory: ID=8, RGB=[249, 61, 251], RGBA=[249, 61, 251, 255]
Posterior-Multimodal: ID=10, RGB=[177, 89, 40], RGBA=[177, 89, 40, 255]
Dorsal-attention: ID=5, RGB=[0, 255, 0], RGBA=[0, 255, 0, 255]
Ventral-Multimodal: ID=11, RGB=[255, 156, 0], RGBA=[255, 156, 0, 255]
Orbito-Affective: ID=12, RGB=[65, 124, 0], RGBA=[65, 124, 0, 255]


In [6]:
network_mapping.keys()

dict_keys(['Visual1', 'Visual2', 'Somatomotor', 'Cingulo-Opercular', 'Language', 'Default', 'Frontoparietal', 'Auditory', 'Posterior-Multimodal', 'Dorsal-attention', 'Ventral-Multimodal', 'Orbito-Affective'])

In [7]:
import pandas as pd

# Load the Glasser parcellation data
glasser_df = pd.read_csv('/home/frank/HBNM/data/Glasser_2016_Labels.csv')

# Load the ColeAnticevic network assignments
with open('/home/frank/HBNM/data/cortex_parcel_network_assignment.txt', 'r') as f:
    network_assignments = [int(line.strip()) for line in f.readlines() if line.strip()]

# Create a comprehensive mapping dataframe
# The network assignment file has 360 entries: first 180 for left hemisphere, next 180 for right hemisphere
parcellation_mapping = []

for idx, row in glasser_df.iterrows():
    parcel_id = row['Parcel']
    
    # Get network assignments for both hemispheres
    left_network_id = network_assignments[parcel_id - 1]  # -1 for 0-based indexing
    right_network_id = network_assignments[parcel_id + 179]  # +179 to get right hemisphere (180-359)
    
    # Find network names from our mapping
    left_network_name = None
    right_network_name = None
    left_network_info = None
    right_network_info = None
    
    for net_name, net_info in network_mapping.items():
        if net_info['id'] == left_network_id:
            left_network_name = net_name
            left_network_info = net_info
        if net_info['id'] == right_network_id:
            right_network_name = net_name
            right_network_info = net_info
    
    # Add left hemisphere entry
    parcellation_mapping.append({
        'Parcel': parcel_id,
        'Hemisphere': 'L',
        'AreaAbbreviation': row['AreaAbbreviation'],
        'AreaName': row['AreaName'],
        'SectionNumber': row['SectionNumber'],
        'Region': row['Region'],
        'NetworkID': left_network_id,
        'NetworkName': left_network_name,
        'NetworkRGB': left_network_info['rgb'] if left_network_info else None,
        'NetworkRGBA': left_network_info['rgba'] if left_network_info else None
    })
    
    # Add right hemisphere entry
    parcellation_mapping.append({
        'Parcel': parcel_id,
        'Hemisphere': 'R',
        'AreaAbbreviation': row['AreaAbbreviation'],
        'AreaName': row['AreaName'],
        'SectionNumber': row['SectionNumber'],
        'Region': row['Region'],
        'NetworkID': right_network_id,
        'NetworkName': right_network_name,
        'NetworkRGB': right_network_info['rgb'] if right_network_info else None,
        'NetworkRGBA': right_network_info['rgba'] if right_network_info else None
    })

# Convert to DataFrame
parcellation_df = pd.DataFrame(parcellation_mapping)

print(f"Created mapping with {len(parcellation_df)} entries")
print(f"Unique networks found: {parcellation_df['NetworkName'].nunique()}")
print("\nFirst few entries:")
print(parcellation_df.head(10))

print("\nNetwork distribution:")
print(parcellation_df['NetworkName'].value_counts())

Created mapping with 360 entries
Unique networks found: 12

First few entries:
   Parcel Hemisphere AreaAbbreviation                        AreaName  \
0       1          L               V1           Primary Visual Cortex   
1       1          R               V1           Primary Visual Cortex   
2       2          L              MST  Medial Superior\nTemporal Area   
3       2          R              MST  Medial Superior\nTemporal Area   
4       3          L               V6               Sixth Visual Area   
5       3          R               V6               Sixth Visual Area   
6       4          L               V2              Second Visual Area   
7       4          R               V2              Second Visual Area   
8       5          L               V3               Third Visual Area   
9       5          R               V3               Third Visual Area   

   SectionNumber                                    Region  NetworkID  \
0              1                            

In [8]:
# Get only the left hemisphere parcels
parcellation_df_L = parcellation_df[parcellation_df['Hemisphere'] == 'L']

# Save the left hemisphere parcellation to disk
save_path = '/home/frank/HBNM/data/parcellation_df_L.csv'
parcellation_df_L.to_csv(save_path, index=False)
print(f"Saved left hemisphere parcellation to {save_path}")

parcellation_df_L

Saved left hemisphere parcellation to /home/frank/HBNM/data/parcellation_df_L.csv


,Parcel,Hemisphere,AreaAbbreviation,AreaName,SectionNumber,Region,NetworkID,NetworkName,NetworkRGB,NetworkRGBA
0,1,L,V1,Primary Visual Cortex,1,Primary_Visual,1,Visual1,"[0, 0, 255]","[0, 0, 255, 255]"
2,2,L,MST,Medial Superior\nTemporal Area,5,MT+_Complex_and_Neighboring_Visual_Areas,2,Visual2,"[100, 0, 255]","[100, 0, 255, 255]"
4,3,L,V6,Sixth Visual Area,3,Dorsal_Stream_Visual,2,Visual2,"[100, 0, 255]","[100, 0, 255, 255]"
6,4,L,V2,Second Visual Area,2,Early_Visual,2,Visual2,"[100, 0, 255]","[100, 0, 255, 255]"
8,5,L,V3,Third Visual Area,2,Early_Visual,2,Visual2,"[100, 0, 255]","[100, 0, 255, 255]"
...,...,...,...,...,...,...,...,...,...,...
350,176,L,STSva,Area STSv anterior,11,Auditory_Association,9,Default,"[255, 0, 0]","[255, 0, 0, 255]"
352,177,L,TE1m,Area TE1 Middle,14,Lateral_Temporal,9,Default,"[255, 0, 0]","[255, 0, 0, 255]"
354,178,L,PI,Para-Insular Area,12,Insular_and_Frontal_Opercular,4,Cingulo-Opercular,"[153, 0, 153]","[153, 0, 153, 255]"
356,179,L,a32pr,Area anterior 32\nprime,19,Anterior_Cingulate_and_Medial_Prefrontal,4,Cingulo-Opercular,"[153, 0, 153]","[153, 0, 153, 255]"


### Timescale per Network

I want a dataframe that contains the timescale for each simulation, for each parcel.

In [9]:
# Create a copy of the left hemisphere dataframe to add timescale data
timescale_parcellation_df = parcellation_df_L.copy()

# Add timescale columns for each model
for model_name, timescales in all_timescales.items():
    # Create a new column for this model's timescales
    # The timescales array is 0-indexed (0-179) but parcel numbers are 1-indexed (1-180)
    timescale_parcellation_df[f'{model_name}_Timescale'] = timescales[timescale_parcellation_df['Parcel'] - 1]

# Display the result
print(f"Created dataframe with {len(timescale_parcellation_df)} rows and {len(timescale_parcellation_df.columns)} columns")
print(f"New timescale columns: {[col for col in timescale_parcellation_df.columns if 'Timescale' in col]}")

timescale_parcellation_df.head()

Created dataframe with 180 rows and 15 columns
New timescale columns: ['Dopamine_Timescale', 'GABAa_Timescale', 'Heterogeneous_Timescale', 'Homogeneous_Timescale', 'NMDA_Timescale']


,Parcel,Hemisphere,AreaAbbreviation,AreaName,SectionNumber,Region,NetworkID,NetworkName,NetworkRGB,NetworkRGBA,Dopamine_Timescale,GABAa_Timescale,Heterogeneous_Timescale,Homogeneous_Timescale,NMDA_Timescale
0,1,L,V1,Primary Visual Cortex,1,Primary_Visual,1,Visual1,"[0, 0, 255]","[0, 0, 255, 255]",0.164892,0.052490,3.248283,0.300983,0.096923
2,2,L,MST,Medial Superior\nTemporal Area,5,MT+_Complex_and_Neighboring_Visual_Areas,2,Visual2,"[100, 0, 255]","[100, 0, 255, 255]",0.571031,1.529641,0.378464,0.676469,2.914685
4,3,L,V6,Sixth Visual Area,3,Dorsal_Stream_Visual,2,Visual2,"[100, 0, 255]","[100, 0, 255, 255]",0.408393,0.396936,0.282154,0.412639,16.444242
6,4,L,V2,Second Visual Area,2,Early_Visual,2,Visual2,"[100, 0, 255]","[100, 0, 255, 255]",0.050000,0.183480,0.163554,0.050000,0.152871
8,5,L,V3,Third Visual Area,2,Early_Visual,2,Visual2,"[100, 0, 255]","[100, 0, 255, 255]",0.224204,0.277493,0.225198,0.350792,181.475195


### Timescale data exploration

In [10]:
# Get all timescale columns
timescale_cols = [col for col in timescale_parcellation_df.columns if 'Timescale' in col]

print("=== BASIC SUMMARY STATISTICS FOR EACH MODEL ===")
print(timescale_parcellation_df[timescale_cols].describe())

print("\n=== TIMESCALE RANGE FOR EACH MODEL ===")
for col in timescale_cols:
    model_name = col.replace('_Timescale', '')
    data = timescale_parcellation_df[col]
    print(f"{model_name}:")
    print(f"  Range: {data.min():.6f} - {data.max():.6f}")
    print(f"  Mean ± Std: {data.mean():.6f} ± {data.std():.6f}")
    print(f"  Median: {data.median():.6f}")
    print()

print("=== TIMESCALE STATISTICS BY NETWORK ===")
network_stats = timescale_parcellation_df.groupby('NetworkName')[timescale_cols].agg(['mean', 'std', 'min', 'max'])
print(network_stats.round(4))

print("\n=== EXTREME VALUES ===")
for col in timescale_cols:
    model_name = col.replace('_Timescale', '')
    print(f"\n{model_name} - Highest timescale regions:")
    top_5 = timescale_parcellation_df.nlargest(5, col)[['AreaAbbreviation', 'NetworkName', col]]
    print(top_5.to_string(index=False))
    
    print(f"\n{model_name} - Lowest timescale regions:")
    bottom_5 = timescale_parcellation_df.nsmallest(5, col)[['AreaAbbreviation', 'NetworkName', col]]
    print(bottom_5.to_string(index=False))

print("\n=== NETWORK RANKING BY MEAN TIMESCALE ===")
for col in timescale_cols:
    model_name = col.replace('_Timescale', '')
    network_means = timescale_parcellation_df.groupby('NetworkName')[col].mean().sort_values(ascending=False)
    print(f"\n{model_name} - Networks ranked by mean timescale:")
    for i, (network, mean_ts) in enumerate(network_means.items(), 1):
        print(f"  {i:2d}. {network:<20} {mean_ts:.6f}")

=== BASIC SUMMARY STATISTICS FOR EACH MODEL ===
       Dopamine_Timescale  GABAa_Timescale  Heterogeneous_Timescale  \
count          180.000000       180.000000               180.000000   
mean             1.035655         1.654963                20.323742   
std              0.901393         0.740093                98.212153   
min              0.050000         0.052490                 0.163554   
25%              0.533411         1.211058                 0.923849   
50%              0.747505         1.562903                 1.511121   
75%              1.227148         2.077027                 9.649923   
max              6.240478         3.931571               999.997744   

       Homogeneous_Timescale  NMDA_Timescale  
count             180.000000      180.000000  
mean                1.030597       80.194913  
std                 0.412622      166.834275  
min                 0.050000        0.096923  
25%                 0.704167        4.244016  
50%                 1.038462  

### Correlation with T1/T2 Hierarchy

In [11]:
myelin_map = np.load('/home/frank/HBNM/data/heterogeneity_vectors/linearized/myelin_linearized.npy')
timescale_parcellation_df['Myelin_T1T2'] = myelin_map[0, timescale_parcellation_df['Parcel'] - 1]

# Compute correlations between each model's timescales and myelin values
from scipy.stats import pearsonr, spearmanr

print("=== TIMESCALE vs MYELIN CORRELATIONS ===\n")

correlations = {}
for col in timescale_cols:
    model_name = col.replace('_Timescale', '')
    
    # Compute Pearson and Spearman correlations
    pearson_r, pearson_p = pearsonr(timescale_parcellation_df[col], timescale_parcellation_df['Myelin_T1T2'])
    spearman_r, spearman_p = spearmanr(timescale_parcellation_df[col], timescale_parcellation_df['Myelin_T1T2'])
    
    correlations[model_name] = {
        'pearson_r': pearson_r,
        'pearson_p': pearson_p,
        'spearman_r': spearman_r,
        'spearman_p': spearman_p
    }
    
    print(f"{model_name}:")
    print(f"  Pearson r = {pearson_r:.4f}, p = {pearson_p:.4e}")
    print(f"  Spearman ρ = {spearman_r:.4f}, p = {spearman_p:.4e}")
    print()

# Display correlation results summary
print("=== CORRELATION SUMMARY ===")
print("Model            Pearson r    Spearman ρ   Significant?")
print("-" * 55)
for model_name, corr_data in correlations.items():
    pearson_sig = "***" if corr_data['pearson_p'] < 0.001 else "**" if corr_data['pearson_p'] < 0.01 else "*" if corr_data['pearson_p'] < 0.05 else "n.s."
    spearman_sig = "***" if corr_data['spearman_p'] < 0.001 else "**" if corr_data['spearman_p'] < 0.01 else "*" if corr_data['spearman_p'] < 0.05 else "n.s."
    print(f"{model_name:<15} {corr_data['pearson_r']:>8.4f}    {corr_data['spearman_r']:>8.4f}    P:{pearson_sig} S:{spearman_sig}")

=== TIMESCALE vs MYELIN CORRELATIONS ===

Dopamine:
  Pearson r = 0.1277, p = 8.7581e-02
  Spearman ρ = -0.0402, p = 5.9215e-01

GABAa:
  Pearson r = -0.3478, p = 1.7210e-06
  Spearman ρ = -0.3192, p = 1.2571e-05

Heterogeneous:
  Pearson r = 0.0179, p = 8.1103e-01
  Spearman ρ = -0.5483, p = 1.6147e-15

Homogeneous:
  Pearson r = -0.3610, p = 6.4036e-07
  Spearman ρ = -0.3482, p = 1.6645e-06

NMDA:
  Pearson r = -0.2101, p = 4.6402e-03
  Spearman ρ = -0.5142, p = 1.5494e-13

=== CORRELATION SUMMARY ===
Model            Pearson r    Spearman ρ   Significant?
-------------------------------------------------------
Dopamine          0.1277     -0.0402    P:n.s. S:n.s.
GABAa            -0.3478     -0.3192    P:*** S:***
Heterogeneous     0.0179     -0.5483    P:n.s. S:***
Homogeneous      -0.3610     -0.3482    P:*** S:***
NMDA             -0.2101     -0.5142    P:** S:***
